In [ ]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace, max as spark_max
from faker import Faker
from datetime import datetime
from datetime import timedelta
import random

spark = SparkSession.builder.getOrCreate()
fake = Faker("en_IN")
 

#  Generate new data using Faker (driver-side)
def generate_patient_data(start_id, n: int=10):
    #start_id = get_starting_id()

    data = []

    for i in range(start_id, start_id + n):
        gender = random.choice(["M", "F"])

        data.append((
            f"P{i:0d}",   # unique persistent ID
            fake.first_name_male() if gender == "M" else fake.first_name_female(),
            fake.last_name(),
            gender,
            fake.date_of_birth(minimum_age=18, maximum_age=80).strftime("%Y-%m-%d"),
            random.choice(["Mumbai", "Delhi", "Bangalore", "Hyderabad", "Chennai"])
        ))

    return data


# Generate hospital data 
def generate_hospital_data(start_id, n: int= 10):
    data = []
    
    prefixes = ["City", "Metro", "Global", "Sunrise", "Green Valley", "Lifeline"]
    suffixes = ["Hospital", "Medical Center", "Health Institute", "Clinic"]


    for i in range(start_id, start_id + n):
        hospital_name = f"{random.choice(prefixes)} {random.choice(suffixes)}"

        data.append((
            f"H{i:0d}", 
            hospital_name,
            random.choice(["Mumbai", "Delhi", "Bangalore", "Hyderabad", "Chennai"]), 
            random.choice(range(200, 1500)) 

        ))
    return data


# Generate new data for visit data
def generate_visit_data(start_id, n: int=10):
    data = []

    for i in range(start_id, start_id + n):
        admission_date = fake.date_between(start_date='-60d', end_date='today')

        data.append((
            f"V1{i:0d}", 
            f"P{i:0d}",
            f"H{i:0d}",            
            str( admission_date ),
            str( admission_date + timedelta(days=random.randint(1, 15)) ),
            f"D{i:0d}",
            random.choice(list(range(500, 100000)))

        ))
    return data


# Generate disgnosis data 
# Sample diagnosis codes (ICD-style simplified)


# Severity levels
severities = ["Mild", "Moderate", "Severe", "Critical"]

# Status
statuses = ["Active", "Recovered", "Chronic"]

def generate_diagnosis_data(start_id, n=10):
    data = []

    diagnosis_description = [
        "Essential hypertension",
        "Type 2 diabetes mellitus",
        "Pneumonia",
        "Lower back pain",
        "Gastro-esophageal reflux disease",
        "Urinary tract infection",
        "Major depressive disorder",
        "Asthma",
        "Obesity",
        "Headache"
    ]

    for i in range(start_id, start_id + n):
        diagnosis_desc = random.choice(diagnosis_description)
        #print( diagnosis_desc )

        diagnosis_date = fake.date_between(start_date='-60d', end_date='today')
        resolved_date = diagnosis_date + timedelta(days=random.randint(2, 20))

        data.append((
            f"D{i:0d}",
            f"P{random.randint(1,50)}",
            f"V1{i:0d}", 
            diagnosis_desc,
            random.choice(severities),
            str(diagnosis_date),
            str(resolved_date),
            random.choice(statuses)
        ))

    return data


patient_data = generate_patient_data(15, 10)
visit_data = generate_visit_data(15, 10)
hospital_data = generate_hospital_data(15, 10)
diagnosis_data = generate_diagnosis_data(15, 10)


patient_cols = ["patient_id", "first_name", "last_name", "gender", "dob", "city"]
visit_cols = ["visit_id", "patient_id", "hospital_id", "admission_date", "discharge_date", "diagnosis_code", "cost"]
hospital_cols = ["hospital_id", "hospital_name", "city", "bed_count"]
diagnosis_cols = ["diagnosis_code", "patient_id", "visit_id", "diagnosis_desc","severity", "diagnosis_date", "resolved_date", "status"]

folder_tables = ["patient", "visit", "hospital", "diagnosis"] 

# Generate sample
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
for name in folder_tables:
    if name == 'patient':
        new_df = spark.createDataFrame(patient_data, patient_cols)
    
    elif name == "visit":
        new_df = spark.createDataFrame(visit_data, visit_cols)

    elif name == "hospital":
        new_df = spark.createDataFrame(hospital_data, hospital_cols)
    elif name == "diagnosis":
        new_df = spark.createDataFrame(diagnosis_data, diagnosis_cols)
    else:
        raise ValueError("There is no table named {name}") 

    path = f"Files/healthcare_data/{name}/{name}_raw" 
    new_df.coalesce(1).write.mode('append').option("header", True).csv(path)

    # Get the generated file
    files = mssparkutils.fs.ls(path)
    part_file = [f.path for f in files if "part-" in f.name][0]

    # Final file path
    final_path = f"Files/healthcare_data/{name}/{name}_{timestamp}.csv"

    # Move + rename
    mssparkutils.fs.mv(part_file, final_path, True)

    # Remove folder
    mssparkutils.fs.rm(path, recurse=True)